# Entrenamiento YOLOv8 - Detector de Baches

**Requisitos antes de correr este notebook:**
1. Tener el dataset exportado de Roboflow en formato **YOLOv8** (archivo ZIP)
2. Subir el ZIP a Google Drive
3. En Colab: Entorno de ejecución → Cambiar tipo → **GPU** (T4)

---

In [ ]:
# ── Paso 1: Instalar dependencias ──────────────────────────────
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

In [ ]:
# ── Paso 2: Montar Google Drive y descomprimir dataset ──────────
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# CAMBIA esta ruta al ZIP que subiste a Drive
ZIP_PATH = '/content/drive/MyDrive/baches_dataset.zip'
DATASET_DIR = '/content/dataset'

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

print('Dataset descomprimido en:', DATASET_DIR)
!ls {DATASET_DIR}

In [ ]:
# ── Paso 3: Ver el data.yaml del dataset ───────────────────────
import glob
yaml_files = glob.glob(f'{DATASET_DIR}/**/*.yaml', recursive=True)
print('YAML encontrado:', yaml_files)

YAML_PATH = yaml_files[0]
with open(YAML_PATH) as f:
    print(f.read())

In [ ]:
# ── Paso 4: Descargar modelo base (pre-entrenado en baches) ────
from huggingface_hub import hf_hub_download

modelo_base = hf_hub_download(
    repo_id='keremberke/yolov8m-pothole-segmentation',
    filename='best.pt'
)
print('Modelo base en:', modelo_base)

In [ ]:
# ── Paso 5: ENTRENAR ──────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(modelo_base)   # parte del modelo ya entrenado en baches

results = model.train(
    data=YAML_PATH,
    epochs=50,          # aumentar a 100 si tienes muchas imágenes
    imgsz=640,
    batch=16,
    name='baches_custom',
    patience=10,        # para si no mejora después de 10 epochs
    augment=True,       # augmentación de datos automática
    degrees=5.0,        # rotación leve (baches se ven desde ángulos distintos)
    flipud=0.0,         # no voltear verticalmente (carretera siempre abajo)
    fliplr=0.5,         # voltear horizontal sí (baches a ambos lados)
)

print('\nEntrenamiento completo.')
print('Mejor modelo en:', results.save_dir)

In [ ]:
# ── Paso 6: Evaluar el modelo ─────────────────────────────────
model_best = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = model_best.val()
print(f'mAP50   : {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')

In [ ]:
# ── Paso 7: Guardar el modelo en Google Drive ─────────────────
import shutil

DESTINO = '/content/drive/MyDrive/mi_modelo_baches.pt'
shutil.copy(f'{results.save_dir}/weights/best.pt', DESTINO)
print(f'Modelo guardado en Drive: {DESTINO}')
print('\nDescárgalo y ponlo en la carpeta del proyecto.')
print('Luego en detector_baches.py cambia MODELO_LOCAL a la ruta de ese archivo.')